In [ ]:
!pip install openai spacy scikit-learn numpy tqdm
!python -m spacy download en_core_web_sm
!python -m spacy download en_core_web_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 55.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 3.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
"""
DRIFT: ACOS Evaluation Script
================================
Evaluates the DRIFT framework on ACOS-Laptop and ACOS-Restaurant datasets
using GPT-4o or GPT-4o-mini via the OpenAI API.

HOW TO RUN:
-----------
1. Install dependencies:
   pip install openai spacy scikit-learn numpy tqdm
   python -m spacy download en_core_web_sm
   python -m spacy download en_core_web_lg

2. Set your API key and BASE_URL in the CONFIGURATION section below.

3. Set your data paths and experiment config (see CONFIGURATION section).

4. Run:
   python drift_acos_evaluation.py

OUTPUT:
-------
- Prints Precision / Recall / F1 to console after each run.
- Saves a JSON file with full results + config + metrics.

BUGS FIXED vs previous version:
---------------------------------
1. Arabic comma removed from output format string (caused garbled element order)
2. Output element order fixed to [Aspect, Category, Sentiment, Opinion] everywhere
   (was [Aspect, Category, Opinion, Sentiment] before — would give 0% F1)
3. Instruction no longer says "restaurant domain" for laptop data (domain-flexible)
4. BASE_URL=None by default; only passed to OpenAI() when explicitly set
5. 401 early-exit added: script stops immediately on bad API key
"""

import os
import json
import ast
import time
import spacy
import numpy as np
from tqdm import tqdm
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity
from typing import List, Dict, Tuple, Set


# ==============================================================================
# CONFIGURATION  <-- Edit these values before running
# ==============================================================================

API_KEY = os.environ.get("OPENAI_API_KEY", "YOUR_API_KEY_HERE")

# BASE_URL — set to None for standard OpenAI (api.openai.com).
# If you use a proxy or alternative provider, paste the URL here.
# Common examples:
#   OpenRouter:        "https://openrouter.ai/api/v1"
#   Azure OpenAI:      "https://YOUR_RESOURCE.openai.azure.com/"
#   University proxy:  ask your IT department for the endpoint URL
BASE_URL = None   # or if use a proxy or alternative provider, paste the URL here such as "https://openrouter.ai/api/v1"


# --- Experiment settings ---
NUM_FEW_SHOTS = 1           # Retrieved demonstrations: 0, 1, 5, or 10
MODEL_NAME    = "gpt-4o-mini"  # "gpt-4o-mini"  or  "gpt-4o" -----> Exactly write this models name
DOMAIN        = "laptop"    # "laptop"  or  "restaurant"

# --- Data paths ---
# ACOS-Laptop:    laptop_quad_train.tsv / laptop_quad_test.tsv
# ACOS-Rest:      rest_quad_train.tsv   / rest_quad_test.tsv
TRAIN_DATA_PATH = "laptop_quad_train.tsv"
TEST_DATA_PATH  = "laptop_quad_test.tsv"

# --- Output ---
OUTPUT_FILE = f"{DOMAIN}_DRIFT_{NUM_FEW_SHOTS}shot_{MODEL_NAME}_results.json"

# Delay between API calls (seconds) — increase if you hit rate limits
API_DELAY = 1.5

# ==============================================================================
# SENTIMENT MAP
# ==============================================================================
SENTIMENT_MAP = {0: "negative", 1: "neutral", 2: "positive"}


# ==============================================================================
# DATA LOADING
# ==============================================================================

def parse_acos_line(line: str) -> Tuple[str, List[List[str]]]:
    """
    Parses one line from an ACOS .tsv file.

    File format:
        sentence<TAB>AS,AE CAT SENT OS,OE<TAB>AS,AE CAT SENT OS,OE ...

    Fields:
        AS,AE = aspect token span  (start inclusive, end exclusive; -1,-1 = NULL)
        CAT   = category string    (e.g. LAPTOP#GENERAL)
        SENT  = sentiment integer  (0=negative, 1=neutral, 2=positive)
        OS,OE = opinion token span (start inclusive, end exclusive; -1,-1 = NULL)

    Returns:
        (sentence_str, list_of_quadruples)
        Each quadruple: [aspect_term, category, sentiment_str, opinion_term]
        Returns (None, None) on malformed or empty lines.
    """
    parts = line.strip().split('\t')
    if len(parts) < 2:
        return None, None

    sentence = parts[0].strip()
    if not sentence:
        return None, None

    tokens     = sentence.split()
    quadruples = []

    for quad_str in parts[1:]:
        quad_str = quad_str.strip()
        if not quad_str:
            continue

        segments = quad_str.split()
        if len(segments) != 4:
            continue

        aspect_span_str, category, sentiment_int_str, opinion_span_str = segments

        # Sentiment
        try:
            sentiment_str = SENTIMENT_MAP[int(sentiment_int_str)]
        except (ValueError, KeyError):
            continue

        # Aspect term
        try:
            a_start, a_end = map(int, aspect_span_str.split(','))
            aspect_term = "NULL" if a_start == -1 else " ".join(tokens[a_start:a_end])
        except Exception:
            aspect_term = "NULL"

        # Opinion term
        try:
            o_start, o_end = map(int, opinion_span_str.split(','))
            opinion_term = "NULL" if o_start == -1 else " ".join(tokens[o_start:o_end])
        except Exception:
            opinion_term = "NULL"

        if not category:
            continue

        # IMPORTANT: order must be [aspect, category, sentiment, opinion]
        # This matches the gold label order used in evaluation.
        quadruples.append([aspect_term, category, sentiment_str, opinion_term])

    if not quadruples:
        return None, None

    return sentence, quadruples


def read_acos_file(data_path: str) -> Tuple[List[str], List[List[List[str]]]]:
    """
    Reads an ACOS .tsv file and returns parallel lists of sentences and labels.
    """
    if not os.path.exists(data_path):
        raise FileNotFoundError(
            f"\nData file not found: '{data_path}'\n"
            f"Check your TRAIN_DATA_PATH / TEST_DATA_PATH settings."
        )

    sentences, labels = [], []

    with open(data_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            sentence, quadruples = parse_acos_line(line)
            if sentence is not None and quadruples is not None:
                sentences.append(sentence)
                labels.append(quadruples)

    print(f"  Loaded {len(sentences)} valid samples from '{data_path}'")
    return sentences, labels


# ==============================================================================
# CATEGORY EXTRACTION
# ==============================================================================

def get_unique_categories(labels: List[List[List[str]]]) -> List[str]:
    """Extracts and sorts all unique category strings from combined labels."""
    categories: Set[str] = set()
    for quads in labels:
        for quad in quads:
            if len(quad) == 4 and quad[1]:
                categories.add(quad[1])
    return sorted(categories)


# ==============================================================================
# DEPENDENCY TREE LINEARIZATION
# ==============================================================================

def linearize_dependency_tree(sentence: str, nlp_model) -> str:
    """
    Converts a sentence to a linearized dependency tree string.

    Format per token:  token -> dep_relation -> head
    Returns:  [DEP] triple1; triple2; ... [/DEP]
    """
    doc     = nlp_model(sentence)
    triples = []
    for token in doc:
        if token.is_punct:
            continue
        triples.append(f"{token.text} -> {token.dep_} -> {token.head.text}")

    if not triples:
        return "[DEP] [/DEP]"

    return "[DEP] " + "; ".join(triples) + " [/DEP]"


# ==============================================================================
# FEW-SHOT RETRIEVER
# ==============================================================================

class SemanticRetriever:
    """
    Semantic similarity retriever using spaCy en_core_web_lg vectors.
    Finds the top-k training sentences most similar to a query sentence.
    """

    def __init__(
        self,
        train_sentences: List[str],
        train_labels: List[List[List[str]]],
        nlp_model
    ):
        self.train_sentences = train_sentences
        self.train_labels    = train_labels
        self.nlp_model       = nlp_model

        print(f"  Building retrieval index for {len(train_sentences)} training sentences...")
        self.embeddings = np.array([
            nlp_model(sent).vector
            for sent in tqdm(train_sentences, desc="  Encoding")
        ])
        print("  Retrieval index built.")

    def retrieve(self, query_sentence: str, k: int) -> List[Dict]:
        """Returns top-k most similar training examples. Empty list if k<=0."""
        if k <= 0:
            return []

        query_vec = self.nlp_model(query_sentence).vector.reshape(1, -1)
        sims      = cosine_similarity(query_vec, self.embeddings).flatten()
        top_k_idx = np.argsort(sims)[::-1][:k]

        return [
            {
                "sentence": self.train_sentences[idx],
                "labels":   self.train_labels[idx],
            }
            for idx in top_k_idx
        ]


# ==============================================================================
# PROMPT CONSTRUCTION
# ==============================================================================

def build_instruction_prompt(categories: List[str], domain: str) -> str:
    """
    Builds the system instruction prompt.

    FIX 1: No Arabic comma. Element order is exactly:
            [Aspect Term, Aspect Category, Sentiment Polarity, Opinion Term]
    FIX 2: Uses the correct domain name (laptop / restaurant).
    FIX 3: Instructs the model to use the dependency tree for aspect-opinion linking.
    """
    category_list_str = "\n  - ".join([""] + categories)

    return f"""ROLE:
You are an expert ACOS (Aspect-Category-Opinion-Sentiment) quadruple extraction bot.
Your processing must be precise, systematic, and strictly follow all rules.

TASK DEFINITION:
Your sole task is to extract all sentiment quadruples from a user-provided sentence.
Your response MUST be a valid Python list of lists, with no other text or explanations.
The output format for each quadruple is a list of four strings:
["<Aspect Term>", "<Aspect Category>", "<Sentiment Polarity>", "<Opinion Term>"]

DATASET STRUCTURE ALIGNMENT:
The dataset annotations are provided in a label-based format:
<aspect_span_indices> <CATEGORY> <SENTIMENT_ID> <opinion_span_indices>

- Aspect and opinion terms are NOT given directly as text, but as token index ranges.
- You MUST reconstruct the exact text spans from the sentence using these indices.
- If a span is missing (represented in the dataset), you MUST output "NULL".
- Sentiment is encoded numerically and must be mapped as:
  0 → "negative", 1 → "neutral", 2 → "positive".
- Categories follow a structured format: ENTITY#ATTRIBUTE.
- Your output must always be text-based, NOT index-based.

ELEMENT DEFINITIONS:
1.  Aspect Term: The specific noun or phrase being reviewed.
    - It MUST be a term from the dataset domain (e.g., product components, features, or attributes).
    - Use the exact string "NULL" if the opinion is general or the aspect is implicit.
2. Aspect Category: The general class for the Aspect Term.
   - You MUST choose ONLY from this predefined list:{category_list_str}
3.  Sentiment: The polarity of the opinion.
    - It MUST be one of three strings: "positive", "negative", or "neutral".
4.  Opinion Term: The exact word(s) from the sentence expressing the sentiment.

EXTRACTION RULES:
1.  Concise Spans: The Opinion Term must be the most concise phrase that still captures the full sentiment. Do not include unnecessary words.
2.  Extract All Opinions: You must extract a separate quadruple for every distinct opinion expressed in the sentence.
3.  Implied Sentiment: Analyze comparative and conditional phrases to determine the implied sentiment.
4.  Span Reconstruction: Always extract exact phrases from the sentence, not indices or paraphrases.

STEP BY STEP PROCESS:
To ensure accuracy, think step-by-step before generating the final list:
1.  First, identify all opinion-expressing words/phrases in the sentence.
2.  Second, for each opinion, identify its specific aspect phrase. If none exist, that aspect is "NULL".
3.  Third, determine the sentiment polarity (positive, negative, neutral) and the correct aspect category from the provided list.
4.  Finally, construct the list of all quadruples according to all rules above.

FINAL OUTPUT INSTRUCTION:
Your final output MUST ONLY be a valid Python list of lists.
- Do not include explanations, reasoning, or markdown formatting.
- Do not include extra text before or after the list.
- Each inner list must contain exactly 4 strings.
"""


def build_user_message(
    sentence: str,
    dependency_tree: str,
    few_shot_examples: List[Dict],
) -> str:
    """Builds the user-turn message with optional few-shot examples."""
    parts = []

    if few_shot_examples:
        parts.append("### Examples")
        parts.append("Use these labelled examples as guidance:\n")
        for ex in few_shot_examples:
            parts.append(f"Sentence: {ex['sentence']}")
            parts.append(f"Output: {json.dumps(ex['labels'])}")
            parts.append("")
        parts.append("### Now extract quadruples for the sentence below:\n")

    parts.append(f"Sentence: {sentence}")
    parts.append(f"Dependency Tree: {dependency_tree}")
    parts.append("Output:")

    return "\n".join(parts)


# ==============================================================================
# OUTPUT PARSER
# ==============================================================================

def parse_model_output(output_text: str) -> List[List[str]]:
    """
    Parses the model's raw text output into a list of 4-element quadruples.

    Tries:
    1. Direct JSON parse
    2. Extract outermost [...] block and use ast.literal_eval
    3. Return [] on failure
    """
    if not output_text or not output_text.strip():
        return []

    text = output_text.strip()
    # Strip markdown code fences if present
    text = text.replace("```python", "").replace("```json", "").replace("```", "").strip()

    # Attempt 1: JSON
    try:
        data = json.loads(text)
        if isinstance(data, list):
            valid = [q for q in data if isinstance(q, (list, tuple)) and len(q) == 4]
            if valid:
                return [list(q) for q in valid]
    except json.JSONDecodeError:
        pass

    # Attempt 2: Find outermost [...] and ast.literal_eval
    start = text.find('[')
    if start == -1:
        return []

    depth, end = 0, -1
    for i in range(start, len(text)):
        if text[i] == '[':
            depth += 1
        elif text[i] == ']':
            depth -= 1
            if depth == 0:
                end = i
                break

    if end == -1:
        return []

    try:
        parsed = ast.literal_eval(text[start:end + 1])
        if isinstance(parsed, list):
            valid = [q for q in parsed if isinstance(q, (list, tuple)) and len(q) == 4]
            return [list(q) for q in valid]
    except Exception:
        pass

    return []


# ==============================================================================
# EVALUATION METRICS
# ==============================================================================

def calculate_metrics(
    all_preds: List[List[List[str]]],
    all_golds: List[List[List[str]]],
) -> Dict[str, float]:
    """
    Micro-averaged Precision, Recall, F1.
    A quadruple is a TP only when all 4 elements match exactly
    (case-insensitive, whitespace-stripped).
    Element order: [aspect, category, sentiment, opinion]
    """
    total_tp = total_fp = total_fn = 0

    for pred_quads, gold_quads in zip(all_preds, all_golds):
        gold_set: Set[Tuple] = {
            tuple(str(x).strip().lower() for x in q)
            for q in gold_quads
            if isinstance(q, (list, tuple)) and len(q) == 4
        }
        pred_set: Set[Tuple] = {
            tuple(str(x).strip().lower() for x in q)
            for q in pred_quads
            if isinstance(q, (list, tuple)) and len(q) == 4
        }

        total_tp += len(gold_set & pred_set)
        total_fp += len(pred_set - gold_set)
        total_fn += len(gold_set - pred_set)

    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    recall    = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    f1        = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0 else 0.0
    )

    return {
        "precision": round(precision * 100, 2),
        "recall":    round(recall    * 100, 2),
        "f1":        round(f1        * 100, 2),
    }


# ==============================================================================
# MAIN EXTRACTOR CLASS
# ==============================================================================

class DRIFTExtractor:
    """
    DRIFT extractor: dependency-tree injection + semantic retrieval-augmented
    few-shot prompting for ACOS quadruple extraction.
    """

    def __init__(self, api_key: str, model_name: str, base_url: str = None):
        if not api_key or api_key == "YOUR_API_KEY_HERE":
            raise ValueError(
                "\n*** NO API KEY ***\n"
                "Paste your API key into API_KEY at the top of the script.\n"
            )

        # Build OpenAI client — only add base_url when explicitly provided
        client_kwargs: Dict = {"api_key": api_key}
        if base_url and base_url.startswith("http"):
            client_kwargs["base_url"] = base_url
            print(f"  Using custom base_url: {base_url}")
        else:
            print("  Using standard OpenAI endpoint (api.openai.com).")

        self.client     = OpenAI(**client_kwargs)
        self.model_name = model_name
        self.nlp_sm     = spacy.load("en_core_web_sm")
        print(f"  DRIFT Extractor ready  |  model: {model_name}")

    def predict(
        self,
        instruction: str,
        sentence: str,
        few_shot_examples: List[Dict],
        max_retries: int = 2,
    ) -> List[List[str]]:
        """Predicts ACOS quadruples for one sentence."""
        dep_tree = linearize_dependency_tree(sentence, self.nlp_sm)
        user_msg = build_user_message(sentence, dep_tree, few_shot_examples)

        messages = [
            {"role": "system", "content": instruction},
            {"role": "user",   "content": user_msg},
        ]

        for attempt in range(max_retries):
            try:
                response = self.client.chat.completions.create(
                    model       = self.model_name,
                    messages    = messages,
                    max_tokens  = 512,
                    temperature = 0.0,
                )
                raw_text = response.choices[0].message.content or ""
                parsed   = parse_model_output(raw_text)

                if not parsed:
                    print(f"    [WARN] Could not parse output (attempt {attempt + 1}):")
                    print(f"    Raw: {raw_text[:300]}")

                return parsed

            except Exception as exc:
                error_str = str(exc)
                print(f"    [API Error] attempt {attempt + 1}/{max_retries}: {exc}")

                # Stop immediately on authentication errors — retrying never helps
                if (
                    "401" in error_str
                    or "invalid_api_key" in error_str
                    or "Incorrect API key" in error_str
                    or "AuthenticationError" in type(exc).__name__
                ):
                    raise SystemExit(
                        "\n\n*** AUTHENTICATION ERROR — SCRIPT STOPPED ***\n"
                        "Your API key is being rejected (HTTP 401).\n\n"
                        "Possible causes:\n"
                        "  1. The key is wrong, expired, or has no credits.\n"
                        "  2. You need BASE_URL for your provider\n"
                        "     (OpenRouter, Azure, university proxy, etc.).\n\n"
                        "Fix:\n"
                        "  - Get a valid key: https://platform.openai.com/account/api-keys\n"
                        "  - Or ask whoever gave you the key for the correct BASE_URL.\n"
                        "  - Update API_KEY and BASE_URL at the top of this script.\n"
                    )

                if attempt < max_retries - 1:
                    wait = 15 * (attempt + 1)
                    print(f"    Waiting {wait}s before retry...")
                    time.sleep(wait)
                else:
                    print("    All retries exhausted. Returning empty list.")
                    return []

        return []


# ==============================================================================
# MAIN
# ==============================================================================

def main():
    print("=" * 60)
    print("  DRIFT ACOS Evaluation")
    print(f"  Model:    {MODEL_NAME}")
    print(f"  Shots:    {NUM_FEW_SHOTS}")
    print(f"  Domain:   {DOMAIN}")
    print(f"  Train:    {TRAIN_DATA_PATH}")
    print(f"  Test:     {TEST_DATA_PATH}")
    print("=" * 60)

    # 1. Load data
    print("\n[1/6] Loading datasets...")
    train_sents, train_labels = read_acos_file(TRAIN_DATA_PATH)
    test_sents,  test_labels  = read_acos_file(TEST_DATA_PATH)

    # 2. Extract categories
    print("\n[2/6] Extracting categories...")
    all_categories = get_unique_categories(train_labels + test_labels)
    print(f"  Found {len(all_categories)} unique categories.")

    # 3. Build instruction
    print("\n[3/6] Building instruction prompt...")
    instruction = build_instruction_prompt(all_categories, DOMAIN)

    # 4. Build retrieval index
    retriever = None
    if NUM_FEW_SHOTS > 0:
        print("\n[4/6] Building semantic retrieval index...")
        try:
            nlp_lg = spacy.load("en_core_web_lg")
        except OSError:
            raise OSError(
                "spaCy 'en_core_web_lg' not found.\n"
                "Run: python -m spacy download en_core_web_lg"
            )
        retriever = SemanticRetriever(train_sents, train_labels, nlp_lg)
    else:
        print("\n[4/6] Skipping retrieval index (0-shot mode).")

    # 5. Init extractor
    print("\n[5/6] Initialising DRIFT extractor...")
    extractor = DRIFTExtractor(
        api_key    = API_KEY,
        model_name = MODEL_NAME,
        base_url   = BASE_URL,
    )

    # 6. Run inference
    print(f"\n[6/6] Running inference on {len(test_sents)} test samples...")
    all_predictions: List[List[List[str]]] = []

    for i, sentence in enumerate(tqdm(test_sents, desc="Predicting")):
        few_shot_examples = (
            retriever.retrieve(sentence, k=NUM_FEW_SHOTS) if retriever else []
        )
        pred = extractor.predict(instruction, sentence, few_shot_examples)
        all_predictions.append(pred)

        if (i + 1) % 50 == 0:
            partial = calculate_metrics(
                all_predictions, test_labels[: len(all_predictions)]
            )
            print(
                f"  [{i+1}/{len(test_sents)}] Running F1: {partial['f1']:.2f}%"
            )

        time.sleep(API_DELAY)

    # 7. Final metrics
    metrics = calculate_metrics(all_predictions, test_labels)

    print("\n" + "=" * 60)
    print("  FINAL RESULTS")
    print(f"  Model:     {MODEL_NAME}")
    print(f"  Shots:     {NUM_FEW_SHOTS}")
    print(f"  Domain:    {DOMAIN}")
    print(f"  Samples:   {len(test_sents)}")
    print("-" * 60)
    print(f"  Precision: {metrics['precision']:.2f}%")
    print(f"  Recall:    {metrics['recall']:.2f}%")
    print(f"  F1-Score:  {metrics['f1']:.2f}%")
    print("=" * 60)

    # 8. Save
    output = {
        "config": {
            "model":      MODEL_NAME,
            "num_shots":  NUM_FEW_SHOTS,
            "domain":     DOMAIN,
            "train_file": TRAIN_DATA_PATH,
            "test_file":  TEST_DATA_PATH,
            "categories": all_categories,
        },
        "metrics": metrics,
        "results": [
            {
                "id":           i,
                "sentence":     test_sents[i],
                "ground_truth": test_labels[i],
                "prediction":   all_predictions[i],
            }
            for i in range(len(test_sents))
        ],
    }

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)

    print(f"\nFull results saved to: '{OUTPUT_FILE}'")
    return metrics


if __name__ == "__main__":
    main()

  DRIFT ACOS Evaluation
  Model:    gpt-4o-mini
  Shots:    1
  Domain:   laptop
  Train:    laptop_quad_train.tsv
  Test:     laptop_quad_test.tsv

[1/6] Loading datasets...
  Loaded 2934 valid samples from 'laptop_quad_train.tsv'
  Loaded 816 valid samples from 'laptop_quad_test.tsv'

[2/6] Extracting categories...
  Found 118 unique categories.

[3/6] Building instruction prompt...

[4/6] Building semantic retrieval index...
  Building retrieval index for 2934 training sentences...


  Encoding: 100%|██████████| 2934/2934 [00:28<00:00, 104.02it/s]


  Retrieval index built.

[5/6] Initialising DRIFT extractor...
  Using custom base_url: https://api.gapgpt.app/v1
  DRIFT Extractor ready  |  model: gpt-4o-mini

[6/6] Running inference on 816 test samples...


Predicting:   0%|          | 0/816 [00:00<?, ?it/s]

    [API Error] attempt 1/2: Request timed out.
    Waiting 15s before retry...
    [API Error] attempt 2/2: Request timed out.
    All retries exhausted. Returning empty list.


Predicting:   0%|          | 1/816 [01:41<22:59:26, 101.55s/it]


KeyboardInterrupt: 